本ノートブックは、原著の内容をベースにしつつ、一部変更を行っております。オリジナルのコードと比較しながら進めたい場合は、[原著のノートブック](https://github.com/Nicolepcx/transformers-the-definitive-guide/blob/049eedb01c3dab9fe4dcc54aada4e31d29d3d449/CH01/ch01_attention_mechanism_variations.ipynb)を併せてご活用ください。

In [1]:
!pip install -q einops==0.8.2

In [2]:
import torch
import torch.nn.functional as F
from einops import rearrange

## マルチヘッドアテンション

In [3]:
def multi_head_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
) -> torch.Tensor:
    """マルチヘッドアテンション（MHA）を計算する。

    MHAでは、各クエリヘッドがそれぞれ対応するキー/バリューヘッドを使用する。

    入力テンソルの形状:
        query: [batch_size, num_heads, query_length, head_dim]
        key:   [batch_size, num_heads, key_length, head_dim]
        value: [batch_size, num_heads, key_length, value_dim]

    クエリとキーのhead_dimは同じである必要がある。
    バリューのvalue_dimはhead_dimと異なっていてもよい。

    Args:
        query: クエリテンソル。
        key: キーテンソル。
        value: バリューテンソル。

    Returns:
        アテンションの出力テンソル。
        形状は
        [batch_size, num_heads, query_length, value_dim]。

    Raises:
        ValueError:
            クエリとキーのhead_dimが異なる場合、
            クエリ、キー、バリューのヘッド数が一致しない場合、
            またはキーとバリューの系列長が異なる場合。
    """
    num_query_heads = query.shape[1]
    num_key_heads = key.shape[1]
    num_value_heads = value.shape[1]
    head_dim = query.shape[-1]

    if query.shape[-1] != key.shape[-1]:
        raise ValueError(
            "クエリとキーのhead_dimは同じである必要があります。"
        )

    if not (
        num_query_heads
        == num_key_heads
        == num_value_heads
    ):
        raise ValueError(
            "クエリ、キー、バリューのヘッド数は同じである必要があります。"
        )

    if key.shape[-2] != value.shape[-2]:
        raise ValueError(
            "キーとバリューの系列長は同じである必要があります。"
        )

    # QK^Tを計算する。
    # query:  [B, H, Sq, D]
    # key:    [B, H, Sk, D]
    # scores: [B, H, Sq, Sk]
    attention_scores = torch.einsum(
        "bhid,bhjd->bhij",
        query,
        key,
    )

    attention_scores *= head_dim**-0.5

    attention_weights = F.softmax(
        attention_scores,
        dim=-1,
    )

    # アテンション重みをバリューに適用する。
    # weights: [B, H, Sq, Sk]
    # value:   [B, H, Sk, Dv]
    # output:  [B, H, Sq, Dv]
    output = torch.einsum(
        "bhij,bhje->bhie",
        attention_weights,
        value,
    )

    return output

In [4]:
query = torch.rand(32, 8, 128, 64)
key = torch.rand(32, 8, 128, 64)
value = torch.rand(32, 8, 128, 64)

In [5]:
a = F.scaled_dot_product_attention(query, key, value)
b = multi_head_attention(query, key, value)
torch.allclose(a, b)

True

## マルチクエリアテンション

In [6]:
def multi_query_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
) -> torch.Tensor:
    """マルチクエリアテンション（MQA）を計算する。

    MQAでは、すべてのクエリヘッドが1つのキー/バリューヘッドを共有する。

    入力テンソルの形状:
        query: [batch_size, num_query_heads, query_length, head_dim]
        key:   [batch_size, 1, key_length, head_dim]
        value: [batch_size, 1, key_length, value_dim]

    クエリとキーのhead_dimは同じである必要がある。
    バリューのvalue_dimはhead_dimと異なっていてもよい。

    Args:
        query: クエリテンソル。
        key: すべてのクエリヘッドで共有するキーテンソル。
        value: すべてのクエリヘッドで共有するバリューテンソル。

    Returns:
        アテンションの出力テンソル。
        形状は
        [batch_size, num_query_heads, query_length, value_dim]。

    Raises:
        ValueError:
            クエリとキーのhead_dimが異なる場合、
            キー/バリューのヘッド数が1でない場合、
            またはキーとバリューの系列長が異なる場合。
    """
    head_dim = query.shape[-1]

    if query.shape[-1] != key.shape[-1]:
        raise ValueError(
            "クエリとキーのhead_dimは同じである必要があります。"
        )

    if key.shape[1] != 1 or value.shape[1] != 1:
        raise ValueError(
            "MQAではキーとバリューのヘッド数は1である必要があります。"
        )

    if key.shape[-2] != value.shape[-2]:
        raise ValueError(
            "キーとバリューの系列長は同じである必要があります。"
        )

    # QK^Tを計算する。
    # query:  [B, Hq, Sq, D]
    # key:    [B, 1,  Sk, D]
    # scores: [B, Hq, Sq, Sk]
    #
    # キーのヘッド数は1なので、すべてのクエリヘッドが
    # 同じキーヘッドを使用する。
    attention_scores = torch.einsum(
        "bhid,bkjd->bhij",
        query,
        key,
    )

    attention_scores *= head_dim**-0.5

    attention_weights = F.softmax(
        attention_scores,
        dim=-1,
    )

    # アテンション重みをバリューに適用する。
    # weights: [B, Hq, Sq, Sk]
    # value:   [B, 1,  Sk, Dv]
    # output:  [B, Hq, Sq, Dv]
    attention_output = torch.einsum(
        "bhij,bkje->bhie",
        attention_weights,
        value,
    )

    return attention_output

In [7]:
query = torch.rand(32, 32, 128, 64)
key = torch.rand(32, 1, 128, 64)
value = torch.rand(32, 1, 128, 64)

In [8]:
a = F.scaled_dot_product_attention(query, key, value, enable_gqa=True)
b = multi_query_attention(query, key, value)
torch.allclose(a, b)

True

## グループ化クエリアテンション

In [9]:
def grouped_query_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
) -> torch.Tensor:
    """グループ化クエリアテンション（GQA）を計算する。

    GQAでは、複数のクエリヘッドが同じキー/バリューヘッドを共有する。

    入力テンソルの形状:
        query: [batch_size, num_query_heads, query_length, head_dim]
        key:   [batch_size, num_kv_heads, key_length, head_dim]
        value: [batch_size, num_kv_heads, key_length, head_dim]

    クエリヘッド数はキー/バリューヘッド数で割り切れる必要がある。
    たとえば、クエリヘッド数が32、キー/バリューヘッド数が8の場合、
    4つのクエリヘッドが1つのキー/バリューヘッドを共有する。

    Args:
        query: クエリテンソル。
        key: キーテンソル。
        value: バリューテンソル。

    Returns:
        Attentionの出力テンソル。
        形状は
        [batch_size, num_query_heads, query_length, head_dim]。

    Raises:
        ValueError:
            クエリヘッド数がキー/バリューヘッド数で割り切れない場合や、
            入力テンソルの形状に整合性がない場合。
    """
    num_query_heads = query.shape[1]
    num_kv_heads = key.shape[1]
    head_dim = query.shape[-1]

    if num_query_heads % num_kv_heads != 0:
        raise ValueError(
            "クエリヘッド数はキー/バリューヘッド数で割り切れる必要があります。"
        )

    if query.shape[-1] != key.shape[-1]:
        raise ValueError(
            "クエリとキーのhead_dimは同じである必要があります。"
        )

    if key.shape[1] != value.shape[1]:
        raise ValueError(
            "キーとバリューのヘッド数は同じである必要があります。"
        )

    if key.shape[-2] != value.shape[-2]:
        raise ValueError(
            "キーとバリューの系列長は同じである必要があります。"
        )

    group_size = num_query_heads // num_kv_heads

    # クエリのヘッド次元をnum_kv_heads × group_sizeに分割する。
    # 同じグループに属するクエリヘッドは、
    # 1つのキー/バリューヘッドを共有する。
    # [B, Hq, Sq, D]
    #     ->
    # [B, Hkv, G, Sq, D]
    query = rearrange(
        query,
        "b (h g) i d -> b h g i d",
        h=num_kv_heads,
        g=group_size,
    )

    # QK^Tを計算する。
    # query: [B, Hkv, G, Sq, D]
    # key:   [B, Hkv,    Sk, D]
    # scores:[B, Hkv, G, Sq, Sk]
    attention_scores = torch.einsum(
        "bhgid,bhjd->bhgij",
        query,
        key,
    )

    attention_scores *= head_dim**-0.5

    attention_weights = F.softmax(
        attention_scores,
        dim=-1,
    )

    # アテンション重みをバリューに適用する。
    # weights: [B, Hkv, G, Sq, Sk]
    # value:   [B, Hkv,    Sk, D]
    # output:  [B, Hkv, G, Sq, D]
    attention_output = torch.einsum(
        "bhgij,bhjd->bhgid",
        attention_weights,
        value,
    )

    # キー/バリューヘッド次元とグループ次元を結合し、
    # 元のクエリヘッド次元に戻す。
    # [B, Hkv, G, Sq, D]
    #     ->
    # [B, Hq, Sq, D]
    output = rearrange(
        attention_output,
        "b h g i d -> b (h g) i d",
    )

    return output

In [10]:
query = torch.rand(32, 32, 128, 64)
key = torch.rand(32, 8, 128, 64)
value = torch.rand(32, 8, 128, 64)

In [11]:
a = F.scaled_dot_product_attention(query, key, value, enable_gqa=True)
b = grouped_query_attention(query, key, value)
torch.allclose(a, b)

True

## クロスアテンション

In [12]:
def cross_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
) -> torch.Tensor:
    """マルチヘッドのクロスアテンションを計算する。

    クロスアテンションでは、クエリとキー/バリューが異なる系列から与えられる。

    入力テンソルの形状:
        query: [batch_size, num_heads, query_length, head_dim]
        key:   [batch_size, num_heads, key_length, head_dim]
        value: [batch_size, num_heads, key_length, value_dim]

    クエリとキーのhead_dimは同じである必要がある。
    バリューのvalue_dimはhead_dimと異なっていてもよい。

    Args:
        query: クエリテンソル。
        key: キーテンソル。
        value: バリューテンソル。

    Returns:
        アテンションの出力テンソル。
        形状は
        [batch_size, num_heads, query_length, value_dim]。
    """
    head_dim = query.shape[-1]

    if query.shape[-1] != key.shape[-1]:
        raise ValueError(
            "クエリとキーのhead_dimは同じである必要があります。"
        )

    if query.shape[1] != key.shape[1]:
        raise ValueError(
            "クエリとキーのヘッド数は同じである必要があります。"
        )

    if key.shape[1] != value.shape[1]:
        raise ValueError(
            "キーとバリューのヘッド数は同じである必要があります。"
        )

    if key.shape[-2] != value.shape[-2]:
        raise ValueError(
            "キーとバリューの系列長は同じである必要があります。"
        )

    # QK^Tを計算する。
    # query:  [B, H, Sq, D]
    # key:    [B, H, Sk, D]
    # scores: [B, H, Sq, Sk]
    attention_scores = torch.einsum(
        "bhid,bhjd->bhij",
        query,
        key,
    )

    attention_scores *= head_dim**-0.5

    attention_weights = F.softmax(
        attention_scores,
        dim=-1,
    )

    # アテンション重みをバリューに適用する。
    # weights: [B, H, Sq, Sk]
    # value:   [B, H, Sk, Dv]
    # output:  [B, H, Sq, Dv]
    output = torch.einsum(
        "bhij,bhje->bhie",
        attention_weights,
        value,
    )

    return output

In [13]:
query = torch.rand(32, 8, 64, 64)
key = torch.rand(32, 8, 128, 64)
value = torch.rand(32, 8, 128, 64)

In [14]:
a = F.scaled_dot_product_attention(query, key, value)
b = cross_attention(query, key, value)
torch.allclose(a, b)

True

## 参考資料

- [torch.nn.functional.scaled_dot_product_attention](https://docs.pytorch.org/docs/2.13/generated/torch.nn.functional.scaled_dot_product_attention.html)